# Customers - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, create_map, lit

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_customers"
target_table = f"{catalog}.silver.olist_customers"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 99441
Number of columns: 12


In [0]:
null_count_customer_id = bronze_df.filter(col("customer_id").isNull()).count()

null_count_customer_unique_id = bronze_df.filter(col("customer_unique_id").isNull()).count()

print("Null count for customer_id:", null_count_customer_id)
print("Null count for customer_unique_id:", null_count_customer_unique_id)

Null count for customer_id: 0
Null count for customer_unique_id: 0


Both customer_id and customer_unique_id do not contain any null values. They both satisfy the non-null requirement for a possible key.

In [0]:
distinct_count_customer_id = bronze_df.select("customer_id").distinct().count()

distinct_count_customer_unique_id = bronze_df.select("customer_unique_id").distinct().count()

print("Distinct count for customer_id:", distinct_count_customer_id)
print("Distinct count for customer_unique_id:", distinct_count_customer_unique_id)

Distinct count for customer_id: 99441
Distinct count for customer_unique_id: 96096


- customer_id has 0 nulls and 99441 distinct values (which matches the # of rows).
- customer_unique_id has 0 nulls but only 96096 distinct values (which is less than the # of rows).
- customer_id is supported as the key for this table.

In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


There are no rescued data.

In [0]:
display(
    bronze_df.withColumn("customer_city_trimmed", trim(col("customer_city")))
    .filter(col("customer_city") != col("customer_city_trimmed"))
    .count()
)

0

In [0]:
display(
    bronze_df.withColumn("customer_state_trimmed", trim(col("customer_state")))
    .filter(col("customer_state") != col("customer_state_trimmed"))
    .count()
)

0

The customer_city and customer_state columns do not contain extra whitespace.

In [0]:
customer_state_null_count = bronze_df.filter(col("customer_state").isNull()).count()

print("Null count for customer_state:", customer_state_null_count)

Null count for customer_state: 0


There are no null values in customer_state.

In [0]:
display(bronze_df.groupBy("customer_state").count())

customer_state,count
SP,41746
SC,3637
MG,11635
PR,5045
RJ,12852
RS,5466
PA,975
GO,2020
ES,2033
BA,3380


## Transform to Silver

In [0]:
brazil_state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AM": "Amazonas",
    "AP": "Amapá",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MG": "Minas Gerais",
    "MS": "Mato Grosso do Sul",
    "MT": "Mato Grosso",
    "PA": "Pará",
    "PB": "Paraíba",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "PR": "Paraná",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RO": "Rondônia",
    "RR": "Roraima",
    "RS": "Rio Grande do Sul",
    "SC": "Santa Catarina",
    "SE": "Sergipe",
    "SP": "São Paulo",
    "TO": "Tocantins"
}

In [0]:
state_map_expr = create_map(
    *[
        item
        for state_code, state_name in brazil_state_map.items()
        for item in (lit(state_code), lit(state_name))
    ]
)

silver_df = bronze_df.withColumn(
    "customer_state_full",
    state_map_expr[col("customer_state")]
)

In [0]:
display(
    silver_df.select("customer_state", "customer_state_full").distinct()
)

customer_state,customer_state_full
SP,São Paulo
SC,Santa Catarina
MG,Minas Gerais
PR,Paraná
RJ,Rio de Janeiro
RS,Rio Grande do Sul
PA,Pará
GO,Goiás
ES,Espírito Santo
BA,Bahia


## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- customer_state_full: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,customer_state_full
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,Santa Catarina
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,São Paulo
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,Minas Gerais
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,Paraná
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,null,/Volumes/ecommerce_dev/landing/raw_files/olist/customers/olist_customers_dataset.csv,2026-08-02T21:27:38.000Z,2026-08-02T22:32:28.544Z,2acff721-58a2-45ca-b68e-54d6ec1b6895,olist,customers,Minas Gerais


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 99441
Silver row count: 99441
